### RUNNING LOCAL DEEPSEEK-R1-1.5B MODEL USING OLLAMA

In [1]:
import requests
import json

url = "http://localhost:11434/api/generate"

data = {
    "model": "deepseek-r1:1.5b",
    "prompt": "Can you explain what transformers are in the context of machine learning?",
    "system": "You are a knowledgeable and precise AI assistant with expertise in machine learning and NLP. Provide clear, technical explanations suitable for a developer audience."
}

response = requests.post(
    url, json=data, stream=True
)  # remove the stream=True to get the full response

# check the response status
if response.status_code == 200:
    print("Generated Text:", end=" ", flush=True)
    # Iterate over the streaming response
    for line in response.iter_lines():
        if line:
            # Decode the line and parse the JSON
            decoded_line = line.decode("utf-8")
            result = json.loads(decoded_line)
            # Get the text from the response
            generated_text = result.get("response", "")
            print(generated_text, end="", flush=True)
else:
    print("Error:", response.status_code, response.text)

Generated Text: Transformers are a class of neural network models designed to handle sequence data by leveraging multi-head attention mechanisms. Unlike traditional sequential architectures (e.g., RNNs, LSTMs), Transformers process all tokens simultaneously at each step, enabling them to capture long-range dependencies and adaptively focus on relevant parts of the input through attention masks.

Key characteristics include:
1. **Parallel Processing**: Each layer processes all tokens in a batch, unlike sequential models that handle one token at a time.
2. **Multi-Head Attention**: This mechanism allows the model to simultaneously consider different aspects of the input via multiple attention heads, enhancing its ability to understand complex relationships.
3. **Adaptive Focus**: Transformers dynamically adjust their focus based on data and task requirements, making them highly flexible for various applications.

Today's transformers are prominent in NLP tasks such as text classification

### RAG SETUP USING OLLAMA 

In [2]:
# An article about the release of LLaMA 3 model.
# https://towardsai.net/p/artificial-intelligence/some-technical-notes-about-llama-3#.
ARTICLE = """
Since the debut of the original version, Llama has become one of the foundational blocks of the open source generative AI space. I prefer to use the term “open models,” given that these releases are not completely open source, but that’s just my preference. Last week, the trend in open models became even hotter with the release Llama 3. The release of Llama 3 builds on incredible momentum within the open model ecosystem and brings its own innovations. The 8B and 70B versions of Llama 3 are available, with a 400B version currently being trained. The Llama 3 architecture is based on a decoder-only model and includes a new, highly optimized 128k tokenizer. This is quite notable, given that, with few exceptions, most large language models simply reuse the same tokenizers. The new tokenizer leads to major performance gains. Another area of improvement in the architecture is the grouped query attention, which was already used in Llama 2 but has been enhanced for the larger models. Grouped query attention helps improve inference performance by caching key parameters. Additionally, the context window has also increased.
Training is one area in which Llama 3 drastically improves over its predecessors. The model was trained on 15 trillion tokens, making the corpus quite large for an 8B parameter model, which speaks to the level of optimization Meta achieved in this release. It’s interesting to note that only 5% of the training corpus consisted of non-English tokens. The training infrastructure utilized 16,000 GPUs, achieving a throughput of 400 TFLOPs, which is nothing short of monumental.
Architecture Meta AI’s Llama 3 features a standard, decoder-only transformer structure. Llama 3 introduces a tokenizer equipped with a 128K token vocabulary, which enhances language encoding efficiency, significantly boosting model performance. To enhance the inference capabilities, Llama 3 integrates grouped query attention (GQA) across models sized at 8B and 70B. These models are trained with sequences up to 8,192 tokens long, using a masking technique to prevent self-attention across document boundaries.
1)Tokenizer The latest iteration of Llama 3 showcases an innovative tokenizer. This tokenizer operates with a vocabulary comprising 128K tokens, optimized beyond its predecessors to yield superior inference performance. Notably, the Llama 3–8B model was trained using an impressive 15 trillion tokens, a feat made possible through effective parameter utilization.
2) GQA Grouped-query attention (GQA) ingeniously combines aspects of multi-head attention (MHA) and multi-query attention (MQA) to form an efficient attention mechanism. By caching keys and values from prior tokens, GQA lessens memory demands as batch sizes or context windows expand, thereby streamlining the decoding process in Transformer models.
3) RoPE Llama 3 employs Rotary Positional Encoding (RoPE), a sophisticated encoding mechanism that strikes a balance between absolute positional encodings and relative positional encodings. This method not only retains a fixed embedding for each token but also applies a rotational computation to the vectors, enhancing the model’s attention calculations.
4) KV Cache Key-Value (KV) caching is a technique deployed to speed up the inference in autoregressive models like GPT and Llama. By storing previously computed keys and values, the model reduces repetitive calculations, thus expediting matrix multiplications and enhancing overall efficiency.
Training Meta AI has pre-trained Llama 3 on over 15 trillion tokens gathered from public sources. The training set is seven times larger than that used for Llama 2 and includes a significantly higher volume of code. With more than 5% of the training data consisting of high-quality, non-English content covering over 30 languages, Llama 3 prepares for multilingual applications, although performance in these languages may not equal that in English.
In pursuit of the highest data quality, Meta AI developed sophisticated filtering systems, including heuristic and NSFW filters, semantic deduplication, and text classifiers. These systems were refined using insights from previous model generations, particularly Llama 2, which was instrumental in generating training data for Llama 3’s quality-assurance classifiers. For its largest models, Llama 3 utilizes a trio of parallelization strategies: data, model, and pipeline parallelization. Its most effective setup reaches over 400 TFLOPS per GPU, facilitated by training on 16,000 GPUs simultaneously within two custom-built 24,000 GPU clusters. Meta AI has also innovated a new training stack that automates error detection, handling, and maintenance to optimize GPU utilization.
Llama 3 Instruct In refining its pretrained models for chat applications, Meta AI has employed a hybrid of supervised fine-tuning (SFT), rejection sampling, proximal policy optimization (PPO), and direct preference optimization (DPO). The selection and quality assurance of prompts and preference rankings significantly influence model performance. Moreover, to ensure model safety, these instruction-fine-tuned models undergo rigorous testing, including red-teaming by experts using adversarial prompts to identify and mitigate potential misuse risks.
The Results Llama 3 achieves top-tier performance across leading industry benchmarks like MMLU and CommonSense QA.
Additionally, Meta AI has curated a new, high-quality human evaluation set comprising 1,800 prompts spanning 12 critical use cases. Access to this set is restricted even within Meta AI to prevent potential overfitting by the modeling teams.
An Impressive Model Llama 3 is a very welcome addition to the open model generative AI stack. The initial benchmark results are quite impressive, and the 400B version could rival GPT-4. Distribution is one area where Meta excelled in this release, making Llama 3 available on all major machine learning platforms. It’s been just a few hours, and we are already seeing open source innovations using Llama 3.
"""

In [3]:
import requests
import json

# Combine extracted text and question
prompt = f"Use the following article as the source and answer the question:\n\n{ARTICLE}\n\nHow many parameters LLaMA 3 Models have?"

data = {
    "model": "deepseek-r1:1.5b",
    "system": "You are a helpful assistant",
    "prompt": prompt
}

response = requests.post("http://localhost:11434/api/generate", json=data, stream=True)

if response.status_code == 200:
    print("Answer:", end=" ", flush=True)
    for line in response.iter_lines():
        if line:
            decoded_line = json.loads(line.decode("utf-8"))
            print(decoded_line.get("response", ""), end="", flush=True)
else:
    print("Error:", response.status_code, response.text)

Answer: The article discusses several aspects of Meta's Llama 3 model, including architecture, training details, tokenization, and benchmark results. However, it does not provide an explicit number for the parameters of Llama 3. 

Key points from the text:
1. **Token Count**: The article mentions using a vocabulary size of 128k (102,400) and 768k (7,680,000) tokens.
2. **Training Data**: It states that training on 15 trillion tokens is used, but this does not directly translate to parameter counts.
3. **Parameters**: Meta AI has previously indicated around 400 billion parameters for Llama 3, based on other models like GPT.

In conclusion, while the article provides valuable information about Llama 3's architecture and training, it does not specify the exact number of parameters. Meta's parameter count is typically higher, and further details would require additional sources.

### USING OLLAMA PY SDK FOR REQUESTS 

In [4]:
import ollama

# Create a client
client = ollama.Client()

# Generate a completion
response = client.chat(
    model='deepseek-r1:1.5b',  # or 'mistral', 'gemma'('gemma3:1b',), etc., depending on what's installed
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'What is the capital of France?'}
    ]
)

# Print the result
print(response['message']['content'])



The capital of France is Paris. It is the most central and culturally significant city in the country, known for its rich history, diverse attractions, and numerous institutions that reflect France's rich history.
